# LeetCode 2137 — Pour Water Between Buckets to Make Water Levels Equal bằng Python

Link đề: https://leetcode.com/problems/pour-water-between-buckets-to-make-water-levels-equal/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def equalizeWater(self, buckets: List[int], loss: int) -> float:
        ...
```

## 1. Mục tiêu học

1. Chuyển câu hỏi "mức nước chung **lớn nhất**" thành câu hỏi **kiểm tra khả thi** `check(v)`: "có thể đưa mọi xô về đúng mức `v` không?".
2. Thấy tính **đơn điệu** (mức thấp hơn luôn dễ hơn) ⇒ **tìm nhị phân trên số thực** (binary search on answer).
3. Viết đúng công thức hao hụt: muốn xô nhận được `d` lít thì phải rót `d / (1 − p)` lít (không phải `d·(1 + p)`).
4. Đi xa hơn tìm nhị phân: hàm `f(v)` **tuyến tính từng khúc** ⇒ sắp xếp + giải phương trình bậc nhất cho đáp án **chính xác** O(n log n).
5. Kiểm chứng bằng một **bộ mô phỏng rót nước thật** (có hao hụt) và so sánh số thực với sai số cho phép.

## 2. Đề bài & ví dụ

Có `n` xô nước, xô `i` chứa `buckets[i]` lít. Cho số nguyên `loss` (phần trăm hao hụt).

Muốn mọi xô có **lượng nước bằng nhau**. Có thể rót **bất kỳ lượng nào** (không nhất thiết nguyên) từ xô này sang xô khác; nhưng mỗi lần rót `k` lít thì **đổ mất `loss%` của `k`** — xô nhận chỉ được `k·(1 − loss/100)`.

Trả về **lượng nước lớn nhất** trong mỗi xô sau khi làm bằng nhau. Chấp nhận sai số `10⁻⁵`.

**Ví dụ 1:** `buckets = [1,2,7], loss = 80` → `2.0`. Rót 5 lít từ xô 2 sang xô 0: mất 4, xô 0 nhận 1 ⇒ `[2, 2, 2]`.

**Ví dụ 2:** `buckets = [2,4,6], loss = 50` → `3.5`. Rót 0,5 từ xô 1 sang xô 0 (nhận 0,25) và 2,5 từ xô 2 sang xô 0 (nhận 1,25) ⇒ `[3.5, 3.5, 3.5]`.

**Ví dụ 3:** `buckets = [3,3,3,3], loss = 40` → `3.0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ buckets[i] ≤ 10⁵`, `0 ≤ loss ≤ 99`.

In [1]:
from typing import List
import random, time, math

examples = [([1, 2, 7], 80, 2.0), ([2, 4, 6], 50, 3.5), ([3, 3, 3, 3], 40, 3.0)]
EPS = 1e-5
def close(a, b, eps=EPS):
    return abs(a - b) <= eps

def pour(state, i, j, k, loss):
    # rót k lít từ xô i sang xô j
    assert state[i] >= k - 1e-12
    state[i] -= k
    state[j] += k * (100 - loss) / 100

s = [2.0, 4.0, 6.0]
pour(s, 1, 0, 0.5, 50); print(s)
pour(s, 2, 0, 2.5, 50); print(s)

[2.25, 3.5, 6.0]
[3.5, 3.5, 3.5]


## 3. Kiến thức nền

### 3.1 Hao hụt theo phần trăm

Rót `k` lít, mất `k·p` (với `p = loss/100`), xô nhận `k·(1 − p)`. Ngược lại, muốn xô **nhận** đúng `d` lít thì phải rót `k = d / (1 − p)`. Vì `loss ≤ 99` nên `1 − p ≥ 0,01 > 0` — không bao giờ chia cho 0.

In [2]:
p = 0.8
d = 1.0
k = d / (1 - p)
print(f"muốn nhận {d} lít với loss 80% -> phải rót {k:.4f} lít; kiểm tra: nhận {k * (1 - p):.4f}")
print("sai lầm d*(1+p) =", d * (1 + p), "-> chỉ nhận được", d * (1 + p) * (1 - p))

muốn nhận 1.0 lít với loss 80% -> phải rót 5.0000 lít; kiểm tra: nhận 1.0000
sai lầm d*(1+p) = 1.8 -> chỉ nhận được 0.35999999999999993


### 3.2 Tìm nhị phân trên số thực

Khi hàm `ok(v)` đơn điệu ("đúng với mọi `v` nhỏ, sai với mọi `v` lớn"), tìm ranh giới bằng cách chia đôi đoạn `[lo, hi]`. Với số thực, hoặc dừng khi `hi − lo < ε`, hoặc lặp **cố định** ~60–100 lần (mỗi lần thu hẹp một nửa; 100 lần đã vượt độ chính xác của `float`).

In [3]:
# Ví dụ: tìm căn bậc hai của 2 bằng nhị phân
lo, hi = 0.0, 2.0
for _ in range(60):
    mid = (lo + hi) / 2
    if mid * mid <= 2: lo = mid
    else: hi = mid
print(lo, math.sqrt(2), abs(lo - math.sqrt(2)))

1.414213562373095 1.4142135623730951 2.220446049250313e-16


### 3.3 So sánh số thực có sai số

Không dùng `==` với `float`. Đề chấp nhận sai số `10⁻⁵` ⇒ so sánh bằng `abs(a − b) ≤ 1e-5` hoặc `math.isclose(a, b, abs_tol=1e-5)`.

In [4]:
print(0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-9), close(3.4999999, 3.5))

False True True


### 3.4 Hàm tuyến tính từng khúc

`max(0, x − v)` là hàm của `v`: bằng `x − v` khi `v ≤ x`, bằng 0 khi `v ≥ x` — hai đoạn thẳng nối ở `v = x`. Tổng của nhiều hàm như vậy vẫn tuyến tính trên từng khoảng giữa hai giá trị `buckets` liên tiếp (sau khi sắp xếp).

In [5]:
xs = [1, 2, 7]
for v in [0, 1, 1.5, 2, 4, 7]:
    print(f"v={v:>3}: thừa = {sum(max(0, x - v) for x in xs):>4}, thiếu = {sum(max(0, v - x) for x in xs):>4}")

v=  0: thừa =   10, thiếu =    0
v=  1: thừa =    7, thiếu =    0
v=1.5: thừa =  6.0, thiếu =  0.5
v=  2: thừa =    5, thiếu =    1
v=  4: thừa =    3, thiếu =    5
v=  7: thừa =    0, thiếu =   11


## 4. Ý tưởng nền tảng / trực giác

Giả sử mức cuối là `v`.
- Xô có `x ≥ v` **thừa** `x − v` lít — đây là nước có thể đem đi rót.
- Xô có `x < v` **thiếu** `v − x` lít — muốn nhận đủ phải có người rót cho nó `(v − x)/(1 − p)` lít.

Mức `v` **khả thi** ⇔ `Σ thừa ≥ Σ thiếu / (1 − p)`.

Tính tay ví dụ 1 (`p = 0,8`, `1 − p = 0,2`):

| v | Σ thừa | Σ thiếu | cần rót = thiếu / 0,2 | khả thi? |
|---|---|---|---|---|
| 1 | 0 + 1 + 6 = 7 | 0 | 0 | ✔ |
| 2 | 0 + 0 + 5 = 5 | 1 | 5 | ✔ (vừa khít) |
| 2,5 | 4,5 | 1,5 + 0,5 = 2 | 10 | ✘ |

Mức càng cao thì thừa càng ít, thiếu càng nhiều ⇒ khả thi là một **tiền tố** `[0, v*]`. Đáp án là `v*` = 2.

Chú ý: rót **giữa các xô thừa** với nhau hoặc rót qua trung gian chỉ làm mất thêm nước — không bao giờ có lợi. Mỗi lít nên đi **thẳng** từ xô thừa tới xô thiếu, chỉ chịu hao hụt **một lần**.

In [6]:
def surplus_need(buckets, loss, v):
    a = sum(x - v for x in buckets if x >= v)
    b = sum((v - x) for x in buckets if x < v) * 100 / (100 - loss)
    return a, b

for v in [1, 2, 2.5, 3]:
    a, b = surplus_need([1, 2, 7], 80, v)
    print(f"v={v}: thừa={a:.3f}, cần rót={b:.3f} -> {'khả thi' if a >= b - 1e-12 else 'không'}")

v=1: thừa=7.000, cần rót=0.000 -> khả thi
v=2: thừa=5.000, cần rót=5.000 -> khả thi
v=2.5: thừa=4.500, cần rót=10.000 -> không
v=3: thừa=4.000, cần rót=15.000 -> không


## 5. Cách 1 — Brute Force / baseline: quét mức `v` theo bước nhỏ

**Ý tưởng:** thử `v = 0, δ, 2δ, …` tới `max(buckets)`, giữ `v` lớn nhất còn khả thi. Đúng tới sai số `δ`, nhưng muốn sai số `10⁻⁵` với `max = 10⁵` thì cần `10¹⁰` bước × `n` ⇒ không khả thi. Chỉ dùng được với `max` nhỏ và `δ` vừa phải.

In [7]:
def feasible(buckets, loss, v):
    a = b = 0.0
    for x in buckets:
        if x >= v: a += x - v
        else: b += (v - x) * 100 / (100 - loss)
    return a >= b

def brute_scan(buckets, loss, step=1e-3):
    best, v, hi = 0.0, 0.0, max(buckets)
    while v <= hi + 1e-12:
        if feasible(buckets, loss, v): best = v
        else: break                      # đơn điệu: sau đó đều không khả thi
        v += step
    return best

for b, l, exp in examples:
    got = brute_scan(b, l)
    print(f"{got:.4f} kỳ vọng {exp} ->", "OK (sai số ≤ bước)" if abs(got - exp) <= 1e-3 + 1e-9 else "SAI")

2.0000 kỳ vọng 2.0 -> OK (sai số ≤ bước)
3.5000 kỳ vọng 3.5 -> OK (sai số ≤ bước)
3.0000 kỳ vọng 3.0 -> OK (sai số ≤ bước)


**Độ phức tạp:** O((max/δ)·n). Sai số tối đa `δ`.

**Nút thắt:** ta thử từng mức một cách tuần tự, trong khi `feasible` **đơn điệu** — có thể chia đôi khoảng tìm kiếm mỗi bước.

## 6. Ý tưởng thuật toán / bước đột phá

**Định nghĩa:** `f(v) = Σ_{x ≥ v} (x − v) − Σ_{x < v} (v − x)/(1 − p)`. Khả thi ⇔ `f(v) ≥ 0`.

**Đơn điệu:** khi `v` tăng, mỗi số hạng thừa giảm (hoặc giữ 0), mỗi số hạng thiếu tăng (hoặc giữ 0) ⇒ `f` **giảm ngặt**… cụ thể độ dốc của `f` là `−(số xô ≥ v) − (số xô < v)/(1 − p) ≤ −n < 0`. Vậy `f` giảm ngặt, liên tục, `f(min) ≥ 0`, `f(max) ≤ 0` ⇒ có **đúng một** nghiệm `v*` trong `[min, max]`, và đó là đáp án.

**Hai hướng khai thác:**
1. **Tìm nhị phân** trên `[0, max]` (hoặc `[min, avg]`): `O(n · log(max/ε))`.
2. **Giải chính xác:** sắp xếp `buckets`; trên mỗi khoảng `[x_i, x_{i+1}]`, `f` là hàm bậc nhất `f(v) = (S_hi − c_hi·v) − (c_lo·v − S_lo)/(1 − p)` với `S_hi, c_hi` là tổng/số lượng các xô `≥ x_{i+1}`… ⇒ tìm khoảng có `f` đổi dấu rồi giải `f(v) = 0`.

**Cận cho nhị phân:** đáp án ≥ `min(buckets)` (mức `min` không cần rót gì) và ≤ `trung bình` (nước không sinh ra thêm; với `loss = 0` đáp án đúng bằng trung bình).

In [8]:
b, l = [1, 2, 7], 80
for v in [0, 1, 2, 3, 5, 7]:
    a, need = surplus_need(b, l, v)
    print(f"f({v}) = {a - need:8.3f}")
print("min =", min(b), "| trung bình =", sum(b) / len(b))

f(0) =   10.000
f(1) =    7.000
f(2) =    0.000
f(3) =  -11.000
f(5) =  -33.000
f(7) =  -55.000
min = 1 | trung bình = 3.3333333333333335


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm nhị phân với điều kiện dừng `hi − lo > ε` (kiểu doocs)

In [9]:
def bs_eps(buckets, loss):
    lo, hi = 0.0, float(max(buckets))
    while hi - lo > 1e-6:
        mid = (lo + hi) / 2
        if feasible(buckets, loss, mid): lo = mid
        else: hi = mid
    return lo

for b, l, exp in examples:
    got = bs_eps(b, l)
    print(f"{got:.6f}", "OK" if close(got, exp) else "SAI")

2.000000 OK
3.500000 OK
2.999999 OK


Dùng `ε = 1e-6` (chặt hơn yêu cầu `1e-5`) để chắc chắn. Số vòng ≈ `log₂(10⁵ / 10⁻⁶) ≈ 37`.

### 7.2 Cách 3 — Tìm nhị phân **số vòng cố định** + cận chặt `[min, trung bình]`

Lặp cố định 100 lần: không phụ thuộc so sánh `float`, không lo vòng lặp vô hạn khi `ε` nhỏ hơn độ chính xác của `float`.

In [10]:
def bs_fixed(buckets, loss, iters=100):
    lo, hi = float(min(buckets)), sum(buckets) / len(buckets)
    keep = (100 - loss) / 100
    for _ in range(iters):
        mid = (lo + hi) / 2
        a = b = 0.0
        for x in buckets:
            if x > mid: a += x - mid
            else: b += mid - x
        if a * keep >= b: lo = mid          # nhân thay vì chia
        else: hi = mid
    return lo

for b, l, exp in examples:
    got = bs_fixed(b, l)
    print(f"{got:.6f}", "OK" if close(got, exp) else "SAI")

2.000000 OK
3.500000 OK
3.000000 OK


Mẹo `a·(1 − p) ≥ b` tương đương `a ≥ b/(1 − p)` nhưng tránh phép chia trong vòng lặp. Với `n = 10⁵`, 100 vòng × 10⁵ = 10⁷ phép — chạy được nhưng chậm trong Python thuần; cách 4 nhanh hơn nhiều.

**Độ phức tạp cách 2/3:** O(n · K) với `K` = số vòng (≈ 37–100); bộ nhớ O(1).

### 7.3 Cách 4 — Sắp xếp + giải phương trình bậc nhất (chính xác)

Sắp xếp `x_0 ≤ … ≤ x_{n−1}`, tổng tiền tố `P`. Giả sử đáp án `v ∈ [x_{i−1}, x_i]` (`i` xô đầu thiếu, `n − i` xô sau thừa). Khi đó:

`(S − P_i) − (n − i)·v = (i·v − P_i) / (1 − p)`, với `S = Σ x`, `P_i = x_0 + … + x_{i−1}`.

Giải: `v = ((S − P_i)·(1 − p) + P_i) / ((n − i)·(1 − p) + i)`.

Ta thử `i = 1, 2, …, n` và nhận nghiệm đầu tiên nằm trong `[x_{i−1}, x_i]` (với `x_n = +∞`). Cũng có thể kiểm tra dấu của `f` tại các điểm `x_i` để chọn khoảng.

In [11]:
def exact_sorted(buckets, loss):
    xs = sorted(buckets); n = len(xs); S = sum(xs); keep = (100 - loss) / 100
    pre = 0.0
    for i in range(1, n + 1):                 # i xô đầu là "thiếu"
        pre += xs[i - 1]
        v = ((S - pre) * keep + pre) / ((n - i) * keep + i)
        upper = xs[i] if i < n else float("inf")
        if xs[i - 1] - 1e-9 <= v <= upper + 1e-9:
            return v
    return float(xs[0])                       # không xảy ra (luôn có nghiệm)

for b, l, exp in examples:
    got = exact_sorted(b, l)
    print(f"{got:.10f}", "OK" if close(got, exp) else "SAI")

2.0000000000 OK
3.5000000000 OK
3.0000000000 OK


Vì sao có thể chọn "nghiệm đầu tiên nằm trong khoảng"? `f` giảm ngặt và liên tục nên có đúng một nghiệm `v*`. Trên khoảng chứa `v*`, công thức bậc nhất của khoảng đó cho đúng `v*`. Trên các khoảng khác, nghiệm của đường thẳng kéo dài (nếu có) sẽ nằm **ngoài** khoảng (vì trên khoảng đó `f` không đổi dấu, hoặc chỉ bằng 0 ở đầu mút — khi đó cũng chính là `v*`). Ta thêm sai số `1e-9` để không lỡ nghiệm nằm đúng ở đầu mút do làm tròn.

**Độ phức tạp cách 4:** O(n log n) (sắp xếp) + O(n), bộ nhớ O(n). Kết quả **chính xác** tới sai số làm tròn của `float`.

## 8. TRACE từng bước

### 8.1 Trace tìm nhị phân (cách 3) trên ví dụ 2 — 12 vòng đầu

In [12]:
def trace_bs(buckets, loss, iters=12):
    lo, hi = float(min(buckets)), sum(buckets) / len(buckets); keep = (100 - loss) / 100
    print(f"khởi tạo lo={lo}, hi={hi}")
    for it in range(1, iters + 1):
        mid = (lo + hi) / 2
        a = sum(x - mid for x in buckets if x > mid); b = sum(mid - x for x in buckets if x <= mid)
        ok = a * keep >= b
        print(f"vòng {it:>2}: mid={mid:.6f} thừa={a:.4f} thừa*(1-p)={a*keep:.4f} thiếu={b:.4f} -> {'khả thi, lo=mid' if ok else 'không, hi=mid'}")
        if ok: lo = mid
        else: hi = mid
    return lo

trace_bs([2, 4, 6], 50)

khởi tạo lo=2.0, hi=4.0
vòng  1: mid=3.000000 thừa=4.0000 thừa*(1-p)=2.0000 thiếu=1.0000 -> khả thi, lo=mid
vòng  2: mid=3.500000 thừa=3.0000 thừa*(1-p)=1.5000 thiếu=1.5000 -> khả thi, lo=mid
vòng  3: mid=3.750000 thừa=2.5000 thừa*(1-p)=1.2500 thiếu=1.7500 -> không, hi=mid
vòng  4: mid=3.625000 thừa=2.7500 thừa*(1-p)=1.3750 thiếu=1.6250 -> không, hi=mid
vòng  5: mid=3.562500 thừa=2.8750 thừa*(1-p)=1.4375 thiếu=1.5625 -> không, hi=mid
vòng  6: mid=3.531250 thừa=2.9375 thừa*(1-p)=1.4688 thiếu=1.5312 -> không, hi=mid
vòng  7: mid=3.515625 thừa=2.9688 thừa*(1-p)=1.4844 thiếu=1.5156 -> không, hi=mid
vòng  8: mid=3.507812 thừa=2.9844 thừa*(1-p)=1.4922 thiếu=1.5078 -> không, hi=mid
vòng  9: mid=3.503906 thừa=2.9922 thừa*(1-p)=1.4961 thiếu=1.5039 -> không, hi=mid
vòng 10: mid=3.501953 thừa=2.9961 thừa*(1-p)=1.4980 thiếu=1.5020 -> không, hi=mid
vòng 11: mid=3.500977 thừa=2.9980 thừa*(1-p)=1.4990 thiếu=1.5010 -> không, hi=mid
vòng 12: mid=3.500488 thừa=2.9990 thừa*(1-p)=1.4995 thiếu=1.5005 -> kh

3.5

### 8.2 Trace cách 4 (giải chính xác) trên ví dụ 2

`xs = [2, 4, 6]`, `S = 12`, `keep = 0,5`.

| i | P_i | v = ((S−P_i)·keep + P_i)/((n−i)·keep + i) | khoảng [x_{i−1}, x_i] | nhận? |
|---|---|---|---|---|
| 1 | 2 | (10·0,5 + 2)/(2·0,5 + 1) = 7/2 = 3,5 | [2, 4] | ✔ |

In [13]:
def trace_exact(buckets, loss):
    xs = sorted(buckets); n = len(xs); S = sum(xs); keep = (100 - loss) / 100; pre = 0
    for i in range(1, n + 1):
        pre += xs[i - 1]
        v = ((S - pre) * keep + pre) / ((n - i) * keep + i)
        upper = xs[i] if i < n else float("inf")
        inside = xs[i - 1] - 1e-9 <= v <= upper + 1e-9
        print(f"i={i}: P_i={pre}, v={v:.6f}, khoảng [{xs[i-1]}, {upper}] -> {'NHẬN' if inside else 'loại'}")
        if inside: return v

trace_exact([2, 4, 6], 50)
print()
trace_exact([0, 10, 10, 100], 30)

i=1: P_i=2, v=3.500000, khoảng [2, 4] -> NHẬN

i=1: P_i=0, v=27.096774, khoảng [0, 10] -> loại
i=2: P_i=10, v=25.588235, khoảng [10, 10] -> loại
i=3: P_i=20, v=24.324324, khoảng [10, 100] -> NHẬN


24.324324324324323

### 8.3 Mô phỏng rót nước thật để kiểm chứng đáp án

Dựng hàm rót thật: đi hai con trỏ trên danh sách xô thừa và xô thiếu, rót từng phần (mỗi lít chỉ đi thẳng một lần). Kiểm tra cuối cùng mọi xô ≈ `v`.

In [14]:
def simulate(buckets, loss, v):
    state = [float(x) for x in buckets]; keep = (100 - loss) / 100
    givers = [i for i, x in enumerate(state) if x > v]
    takers = [i for i, x in enumerate(state) if x < v]
    log = []; gi = 0
    for t in takers:
        while state[t] < v - 1e-12 and gi < len(givers):
            g = givers[gi]
            can_give = state[g] - v
            need_pour = (v - state[t]) / keep
            k = min(can_give, need_pour)
            pour(state, g, t, k, loss); log.append((g, t, round(k, 6)))
            if state[g] <= v + 1e-12: gi += 1
    return state, log

for b, l, exp in examples:
    ans = exact_sorted(b, l)
    st, log = simulate(b, l, ans)
    print(f"{b}, loss={l}: v={ans:.4f}, các lần rót (từ, tới, lít)={log}, kết quả={[round(x, 6) for x in st]}")

[1, 2, 7], loss=80: v=2.0000, các lần rót (từ, tới, lít)=[(2, 0, 5.0)], kết quả=[2.0, 2.0, 2.0]
[2, 4, 6], loss=50: v=3.5000, các lần rót (từ, tới, lít)=[(1, 0, 0.5), (2, 0, 2.5)], kết quả=[3.5, 3.5, 3.5]
[3, 3, 3, 3], loss=40: v=3.0000, các lần rót (từ, tới, lít)=[], kết quả=[3.0, 3.0, 3.0, 3.0]


## 9. Code nộp LeetCode

Chọn cách 4 (chính xác, O(n log n)). Bản tìm nhị phân (cách 3) cũng nộp được.

In [15]:
from typing import List

class Solution:
    def equalizeWater(self, buckets: List[int], loss: int) -> float:
        xs = sorted(buckets)
        n = len(xs)
        total = sum(xs)
        keep = (100 - loss) / 100          # tỉ lệ nước đến được nơi nhận
        pre = 0                            # tổng i xô nhỏ nhất (các xô thiếu)
        for i in range(1, n + 1):
            pre += xs[i - 1]
            v = ((total - pre) * keep + pre) / ((n - i) * keep + i)
            upper = xs[i] if i < n else float("inf")
            if xs[i - 1] - 1e-9 <= v <= upper + 1e-9:
                return v
        return float(xs[0])

In [16]:
sol = Solution()
for b, l, exp in examples:
    got = sol.equalizeWater(b, l)
    print(f"{got:.5f} | kỳ vọng {exp:.5f} |", "OK" if close(got, exp) else "SAI")

2.00000 | kỳ vọng 2.00000 | OK
3.50000 | kỳ vọng 3.50000 | OK
3.00000 | kỳ vọng 3.00000 | OK


Lưu ý khi `i` xô đầu là "thiếu" nhưng có xô bằng đúng `v` (ví dụ 1: xô 2 lít bằng đáp án 2): xô đó thừa 0 / thiếu 0 — đặt nó vào nhóm nào cũng cho cùng phương trình tại `v`, nên công thức vẫn đúng.

## 10. Vì sao thuật toán đúng

**Điều kiện cần.** Giả sử đạt mức chung `v`. Gọi `G` = tập xô thừa (`x > v`), `T` = tập xô thiếu. Mỗi xô `t ∈ T` cuối cùng nhận ròng `v − x_t`. Nước đổ vào `t` (trừ phần `t` tự rót đi rồi nhận lại — vô ích) phải xuất phát từ đâu đó; mỗi lít rót đi chỉ tới nơi `keep` lít ⇒ tổng lượng **rót ra** ≥ `Σ_T (v − x_t)/keep`. Mà lượng nước **có thể rót ra** mà vẫn giữ các xô thừa ở mức `v` (và không làm xô nào xuống dưới `v` vĩnh viễn) không vượt quá `Σ_G (x_g − v)` (bảo toàn: nước chỉ mất đi, không sinh ra). Vậy cần `Σ_G (x − v) ≥ Σ_T (v − x)/keep`, tức `f(v) ≥ 0`.

**Điều kiện đủ.** Nếu `f(v) ≥ 0`, bộ mô phỏng (mục 8.3) rót trực tiếp từ xô thừa sang xô thiếu đủ để lấp mọi xô thiếu; phần thừa dư (nếu `f(v) > 0`) có thể đem rót sang một xô bất kỳ rồi rót ngược lại đến khi mất hết — hoặc đơn giản là ta chỉ quan tâm mức **lớn nhất**, nơi `f(v*) = 0` nên không có dư.

**Đáp án = nghiệm duy nhất của `f`:** `f` liên tục, giảm ngặt (độ dốc ≤ −n), `f(min) ≥ 0` và `f(avg) ≤ 0` (tại trung bình, thừa = thiếu, nên `f = thiếu − thiếu/keep ≤ 0`). Tìm nhị phân hội tụ về `v*`; cách 4 giải đúng phương trình trên khoảng chứa `v*`. ∎

## 11. Phân tích Time / Space Complexity

`n = len(buckets)`, `M = max(buckets) ≤ 10⁵`.

| Cách | Time | Space | Độ chính xác |
|---|---|---|---|
| 1. Quét bước δ | O((M/δ)·n) | O(1) | δ |
| 2. Nhị phân theo ε | O(n·log(M/ε)) | O(1) | ε |
| 3. Nhị phân cố định K vòng | O(n·K) | O(1) | (avg−min)/2^K |
| 4. Sắp xếp + phương trình | O(n log n) | O(n) | làm tròn float |

In [17]:
rng = random.Random(2137)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for name, f in [("bs_eps", bs_eps), ("bs_fixed(60)", lambda b, l: bs_fixed(b, l, 60)),
                ("exact_sorted", exact_sorted), ("Solution", Solution().equalizeWater)]:
    t = time.perf_counter(); r = f(big, 37); dt = time.perf_counter() - t
    print(f"{name:<13} -> {r:.6f}  {dt*1000:.0f} ms")

bs_eps        -> 44228.265843  199 ms


bs_fixed(60)  -> 44228.265843  242 ms
exact_sorted  -> 44228.265843  23 ms
Solution      -> 44228.265843  24 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Các kỳ vọng tự dựng được tính tay (đối chiếu lại đề gốc nếu có Premium):
- `[5], loss bất kỳ` → 5 (một xô).
- `loss = 0` → trung bình.
- `[0, 0, 10], loss = 0` → 10/3.
- `[0, 10], loss = 50` → `v` thỏa `10 − v = v/0,5` ⇒ `v = 10/3`.
- `[7, 7, 7, 8], loss = 99` → `8 − v = 3(v − 7)/0,01` ⇒ `v = 2108/301 ≈ 7,00332` (hao hụt cực lớn nên gần như không nâng được mức).

In [18]:
ALL = {"bs_eps": bs_eps, "bs_fixed": bs_fixed, "exact_sorted": exact_sorted, "Solution": Solution().equalizeWater}
edge = [(b, l, e) for b, l, e in examples] + [
    ([5], 99, 5.0),
    ([1, 2, 3, 4], 0, 2.5),
    ([0, 0, 10], 0, 10 / 3),
    ([0, 10], 50, 10 / 3),
    ([0, 0, 0], 99, 0.0),
    ([0, 100000], 99, 100000 * 0.01 / 1.01),     # 100000 - v = v / 0.01
    ([7, 7, 7, 8], 99, 2108 / 301),              # 8 - v = 3(v - 7)/0.01  =>  v = 2108/301
]
for b, l, e in edge:
    for name, f in ALL.items():
        got = f(list(b), l)
        assert close(got, e), (name, b, l, got, e)
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên: các cách với nhau, và với điều kiện `f(v*) ≈ 0`, và bằng mô phỏng rót thật

In [19]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 12)
    b = [rng.randint(0, rng.choice([5, 100, 100000])) for _ in range(n)]
    l = rng.randint(0, 99)
    ref = exact_sorted(b, l)
    for name, f in ALL.items():
        assert close(f(b, l), ref, 1e-5 * max(1, ref)), (name, b, l)
    # f(ref) ≈ 0, và mức cao hơn một chút thì không khả thi
    a, need = surplus_need(b, l, ref)
    assert abs(a - need) <= 1e-6 * max(1, sum(b))
    if max(b) > min(b):
        assert not feasible(b, l, ref + 1e-3)
    st, _ = simulate(b, l, ref)
    assert all(abs(x - ref) <= 1e-6 * max(1, ref) for x in st), (b, l, ref, st)
print("1500 case ngẫu nhiên OK (bao gồm mô phỏng rót thật)")

1500 case ngẫu nhiên OK (bao gồm mô phỏng rót thật)


### 12.3 Tính chất: đáp án nằm giữa `min` và trung bình; tăng `loss` thì đáp án không tăng

In [20]:
rng = random.Random(7)
for _ in range(300):
    b = [rng.randint(0, 1000) for _ in range(rng.randint(1, 30))]
    prev = float("inf")
    for l in [0, 10, 50, 90, 99]:
        v = Solution().equalizeWater(b, l)
        assert min(b) - 1e-9 <= v <= sum(b) / len(b) + 1e-9
        assert v <= prev + 1e-9
        prev = v
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Áp hao hụt sai phía (chia phần **thừa** thay vì phần **thiếu**)

In [21]:
def wrong_side(buckets, loss):
    lo, hi = 0.0, float(max(buckets))
    for _ in range(100):
        mid = (lo + hi) / 2
        a = sum(x - mid for x in buckets if x >= mid) * 100 / (100 - loss)   # SAI: phóng to nước thừa
        b = sum(mid - x for x in buckets if x < mid)
        if a >= b: lo = mid
        else: hi = mid
    return lo

print("bản sai :", round(wrong_side([1, 2, 7], 80), 5))
print("bản đúng:", round(Solution().equalizeWater([1, 2, 7], 80), 5))

bản sai : 5.42857
bản đúng: 2.0


**Vì sao sai:** hao hụt làm nước **ít đi**; phải là "thừa × (1 − p) ≥ thiếu" (hay "thiếu / (1 − p) ≤ thừa"). Bản sai làm như nước thừa được **nhân lên**, cho đáp án vượt cả trung bình — vô lý.

### Lỗi 2 — Dùng `d·(1 + p)` cho lượng cần rót

In [22]:
def wrong_one_plus_p(buckets, loss):
    p = loss / 100; lo, hi = 0.0, float(max(buckets))
    for _ in range(100):
        mid = (lo + hi) / 2
        a = sum(x - mid for x in buckets if x >= mid)
        b = sum(mid - x for x in buckets if x < mid) * (1 + p)                 # SAI
        if a >= b: lo = mid
        else: hi = mid
    return lo

print("bản sai :", round(wrong_one_plus_p([1, 2, 7], 80), 5))
print("bản đúng:", round(Solution().equalizeWater([1, 2, 7], 80), 5))

bản sai : 2.69565
bản đúng: 2.0


**Vì sao sai:** rót `d(1 + p)` thì chỉ nhận `d(1 + p)(1 − p) = d(1 − p²) < d`. Đúng phải là `d / (1 − p)`. Với `p = 0,8`: cần 5 lít cho mỗi lít nhận, không phải 1,8.

### Lỗi 3 — Cận trên tìm nhị phân quá nhỏ (`hi = min(buckets)`)

In [23]:
def wrong_hi(buckets, loss):
    lo, hi = 0.0, float(min(buckets))           # SAI: đáp án thường > min
    for _ in range(100):
        mid = (lo + hi) / 2
        if feasible(buckets, loss, mid): lo = mid
        else: hi = mid
    return lo

print("bản sai :", round(wrong_hi([2, 4, 6], 50), 5))
print("bản đúng:", round(Solution().equalizeWater([2, 4, 6], 50), 5))

bản sai : 2.0
bản đúng: 3.5


**Sửa:** `hi` phải chắc chắn ≥ đáp án: `max(buckets)` hoặc trung bình. `lo` phải chắc chắn khả thi: `0` hoặc `min(buckets)`.

### Lỗi 4 — Dùng `loss` như phân số (quên chia 100) / chia nguyên

In [24]:
def wrong_percent(buckets, loss):
    lo, hi = 0.0, float(max(buckets))
    for _ in range(100):
        mid = (lo + hi) / 2
        a = sum(x - mid for x in buckets if x >= mid)
        b = sum(mid - x for x in buckets if x < mid) * 100 // (100 - loss)      # SAI: // làm tròn xuống
        if a >= b: lo = mid
        else: hi = mid
    return lo

print("loss 50 (100/50 = 2 nguyên, trùng hợp đúng): sai", round(wrong_percent([2, 4, 6], 50), 5), "| đúng", round(Solution().equalizeWater([2, 4, 6], 50), 5))
print("loss 30 (100/70 không nguyên)             : sai", round(wrong_percent([2, 4, 6], 30), 5), "| đúng", round(Solution().equalizeWater([2, 4, 6], 30), 5))

loss 50 (100/50 = 2 nguyên, trùng hợp đúng): sai 3.5 | đúng 3.5
loss 30 (100/70 không nguyên)             : sai 4.0 | đúng 3.75


**Vì sao sai:** `//` là chia lấy phần nguyên (với float cũng làm tròn xuống) ⇒ lượng cần rót bị **ước lượng thấp** ⇒ đáp án quá cao. Với `loss = 50` hay `80`, `100/(100 − loss)` tình cờ là số nguyên nên ví dụ của đề không phát hiện được lỗi; `loss = 30` thì lộ ngay (bản sai coi như không hao hụt, ra đúng trung bình 4). **Sửa:** dùng `/`.

### Lỗi 5 — Vòng `while hi - lo > ε` với `ε` quá nhỏ (vòng lặp vô hạn)

Với `ε = 1e-15` và đáp án ~10⁵, khoảng cách hai số `float` liền kề quanh 10⁵ là ~1,5·10⁻¹¹ — lớn hơn `ε`. Khi `lo` và `hi` là hai float liền kề, `mid` bằng một trong hai ⇒ khoảng không bao giờ nhỏ hơn `ε` ⇒ lặp mãi. Ta minh họa có giới hạn số vòng:

In [25]:
def eps_loop(buckets, loss, eps, cap=2000):
    lo, hi = 0.0, float(max(buckets)); it = 0
    while hi - lo > eps and it < cap:
        mid = (lo + hi) / 2
        if feasible(buckets, loss, mid): lo = mid
        else: hi = mid
        it += 1
    return it

print("eps=1e-6 :", eps_loop([0, 100000], 50, 1e-6), "vòng")
print("eps=1e-15:", eps_loop([0, 100000], 50, 1e-15), "vòng (chạm giới hạn an toàn = lặp vô hạn nếu không chặn)")
print("khoảng cách float quanh 1e5:", math.ulp(1e5))

eps=1e-6 : 37 vòng
eps=1e-15: 2000 vòng (chạm giới hạn an toàn = lặp vô hạn nếu không chặn)
khoảng cách float quanh 1e5: 1.4551915228366852e-11


**Sửa:** dùng `ε` hợp lý (`1e-6`–`1e-7`) hoặc lặp **cố định** 60–100 vòng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giá trị **lớn nhất/nhỏ nhất** sao cho **có thể** làm được X" + kiểm tra X dễ + đơn điệu ⇒ **binary search on answer**.
- Đáp án số thực, sai số cho phép ⇒ nhị phân số thực, lặp cố định số vòng.
- Hao hụt theo tỉ lệ: muốn **nhận** `d` thì **gửi** `d / (1 − p)`.
- Khi hàm kiểm tra là **tổng các `max(0, ·)`** ⇒ tuyến tính từng khúc ⇒ có thể giải **chính xác** sau khi sắp xếp.
- Bài cùng họ: 875 Koko Eating Bananas, 1011, 410 Split Array Largest Sum, 774 Minimize Max Distance to Gas Station (nhị phân số thực), 1891.

## 15. Biến thể của bài

### 15.1 Biến thể A — Hao hụt **cố định** `c` lít mỗi lần rót (không theo tỉ lệ)

Khi đó nên rót **ít lần nhất**: mỗi xô thiếu nhận từ đúng một lần rót? Không còn đơn giản — nhưng với giả thiết "mỗi xô thiếu được rót đúng một lần từ một nguồn gộp" thì điều kiện thành `Σ thừa ≥ Σ thiếu + c · (số xô thiếu)`. Vẫn đơn điệu ⇒ tìm nhị phân được (bản mô hình hóa đơn giản).

In [26]:
def equalize_fixed_loss(buckets, c, iters=100):
    lo, hi = float(min(buckets)), float(max(buckets))
    for _ in range(iters):
        mid = (lo + hi) / 2
        a = sum(x - mid for x in buckets if x > mid)
        need = sum(mid - x + c for x in buckets if x < mid)
        if a >= need: lo = mid
        else: hi = mid
    return lo

print(round(equalize_fixed_loss([1, 2, 7], 0), 5), round(equalize_fixed_loss([1, 2, 7], 1), 5))

3.33333 2.66667


### 15.2 Biến thể B — Chỉ được rót từ trái sang phải (xô `i` chỉ rót cho xô `j > i`)

Mức `v` khả thi ⇔ với mọi tiền tố, nước thừa tích lũy (sau hao hụt) đủ bù thiếu hụt tích lũy: quét trái → phải giữ "kho" `bank`: xô thừa thêm `(x − v)·keep` (vì phải rót đi mới dùng được), xô thiếu lấy `v − x`; nếu `bank < 0` ⇒ không khả thi. Vẫn đơn điệu theo `v`.

In [27]:
def equalize_left_to_right(buckets, loss, iters=100):
    keep = (100 - loss) / 100
    def ok(v):
        bank = 0.0
        for x in buckets:
            if x >= v: bank += (x - v) * keep
            else:
                bank -= v - x
                if bank < -1e-12: return False
        return True
    lo, hi = 0.0, float(max(buckets))
    for _ in range(iters):
        mid = (lo + hi) / 2
        if ok(mid): lo = mid
        else: hi = mid
    return lo

print(round(equalize_left_to_right([7, 2, 1], 80), 5), "| ngược chiều:", round(equalize_left_to_right([1, 2, 7], 80), 5))

2.0 | ngược chiều: 1.0


Với `[1, 2, 7]` chỉ rót sang phải thì xô 0 không bao giờ được nhận ⇒ đáp án là 1 (= `buckets[0]`); với `[7, 2, 1]` thì giống bài gốc.

### 15.3 Biến thể C — Hỏi **lượng nước tối thiểu bị mất** để đạt mức chung lớn nhất

Tại `v*`, mọi nước thừa đều đã được rót: lượng rót ra = `Σ thừa`, lượng mất = `Σ thừa · p`. Đơn giản: `mất = Σ buckets − n · v*`.

In [28]:
for b, l, _ in examples:
    v = Solution().equalizeWater(b, l)
    print(b, l, "-> mức", round(v, 5), "| nước mất =", round(sum(b) - len(b) * v, 5))

[1, 2, 7] 80 -> mức 2.0 | nước mất = 4.0
[2, 4, 6] 50 -> mức 3.5 | nước mất = 1.5
[3, 3, 3, 3] 40 -> mức 3.0 | nước mất = 0.0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét bước nhỏ | thử lần lượt | O((M/δ)·n) | O(1) | minh họa |
| 2. Nhị phân ε | chia đôi đến khi đủ hẹp | O(n log(M/ε)) | O(1) | quen thuộc |
| 3. Nhị phân K vòng | chia đôi cố định | O(n·K) | O(1) | **an toàn nhất** về float |
| 4. Sắp xếp + phương trình | f tuyến tính từng khúc | O(n log n) | O(n) | nhanh, chính xác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `equalizeWater([0, 4], 0)` và `equalizeWater([0, 4], 75)`?

**Bài 2 (tính tay):** Với `[1, 5, 9]`, `loss = 50`: viết `f(v)` trên khoảng `[1, 5]` và giải `f(v) = 0`.

**Bài 3 (điền TODO):** Viết `bt3_todo(buckets, loss, v)` trả `True` nếu mức `v` khả thi (hàm `check`), dùng phép nhân thay vì chia.

**Bài 4 (tự viết lại):** Viết `bt4_todo(buckets, loss)` cài tìm nhị phân 100 vòng trên `[min, trung bình]`.

**Bài 5 (mini challenge):** Viết `bt5_todo(buckets, loss, target)` trả về **`loss` lớn nhất** (số nguyên 0..99) sao cho vẫn đạt được mức chung ≥ `target` (hoặc −1 nếu ngay cả `loss = 0` cũng không đạt).

In [29]:
def bt3_todo(buckets, loss, v):
    # TODO: kiểm tra khả thi mức v
    pass

def bt4_todo(buckets, loss):
    # TODO: nhị phân 100 vòng trên [min, avg]
    pass

def bt5_todo(buckets, loss_target_unused, target):
    # TODO: loss lớn nhất để vẫn đạt mức >= target
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `loss = 0` ⇒ trung bình. Mức 2: `loss = 75` ⇒ `4 − v = v / 0,25 = 4v`. Mức 3: `2.0` và `0.8`.

**Bài 2.** Mức 1: trên `[1, 5]`: xô 1 thiếu, xô 5 và 9 thừa. Mức 2: `f(v) = (5 − v) + (9 − v) − (v − 1)/0,5 = 16 − 4v`. Mức 3: `v = 4`.

**Bài 3.** Mức 1: tính thừa `a`, thiếu `b`. Mức 2: khả thi ⇔ `a·(100 − loss) ≥ b·100`. Mức 3: nhân cả hai vế để tránh chia.

**Bài 4.** Mức 1: `lo = min`, `hi = avg`. Mức 2: 100 vòng, `mid` khả thi ⇒ `lo = mid`. Mức 3: trả `lo`.

**Bài 5.** Mức 1: đáp án giảm khi `loss` tăng (mục 12.3) ⇒ đơn điệu theo `loss`. Mức 2: tìm nhị phân **số nguyên** trên `loss ∈ [0, 99]`, hoặc duyệt 100 giá trị. Mức 3: kiểm tra `feasible(buckets, loss, target)` thay vì tính cả đáp án.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2.0`, `0.8`. **Bài 2:** `f(v) = 16 − 4v` ⇒ `v = 4`.

In [30]:
assert close(Solution().equalizeWater([0, 4], 0), 2.0) and close(Solution().equalizeWater([0, 4], 75), 0.8)
assert close(Solution().equalizeWater([1, 5, 9], 50), 4.0)

def bt3(buckets, loss, v):
    a = sum(x - v for x in buckets if x >= v)
    b = sum(v - x for x in buckets if x < v)
    return a * (100 - loss) >= b * 100

def bt4(buckets, loss):
    lo, hi = float(min(buckets)), sum(buckets) / len(buckets)
    for _ in range(100):
        mid = (lo + hi) / 2
        if bt3(buckets, loss, mid): lo = mid
        else: hi = mid
    return lo

def bt5(buckets, _unused, target):
    if not bt3(buckets, 0, target): return -1
    lo, hi = 0, 99                     # loss lớn nhất còn khả thi
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if bt3(buckets, mid, target): lo = mid
        else: hi = mid - 1
    return lo

assert bt3([1, 2, 7], 80, 2) and not bt3([1, 2, 7], 80, 2.01)
assert bt5([1, 2, 7], None, 2) == 80 and bt5([1, 2, 7], None, 4) == -1 and bt5([3, 3], None, 3) == 99
rng = random.Random(1937)
for _ in range(800):
    b = [rng.randint(0, 50) for _ in range(rng.randint(1, 8))]; l = rng.randint(0, 99)
    assert close(bt4(b, l), exact_sorted(b, l))
    tgt = rng.uniform(min(b), max(b))
    exp = -1 if not feasible(b, 0, tgt) else max(L for L in range(100) if feasible(b, L, tgt))
    assert bt5(b, None, tgt) == exp
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
keep = (100 - loss) / 100
def ok(v):                                     # mức v khả thi?
    a = sum(x - v for x in buckets if x > v)   # thừa
    b = sum(v - x for x in buckets if x <= v)  # thiếu
    return a * keep >= b
lo, hi = min(buckets), sum(buckets) / len(buckets)
for _ in range(100):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if ok(mid) else (lo, mid)
return lo
```

**Checklist:**
- [ ] Hao hụt áp vào phía **rót đi**: thừa × (1 − p) ≥ thiếu.
- [ ] Dùng `/` (không `//`), `loss` là phần trăm.
- [ ] `lo` khả thi (min hoặc 0), `hi` ≥ đáp án (trung bình hoặc max).
- [ ] Lặp cố định số vòng hoặc `ε` hợp lý (không quá nhỏ).
- [ ] Test: một xô, `loss = 0` (= trung bình), `loss = 99`.
- [ ] (Premium) Đối chiếu lại đề gốc.